# Qwen3.6-35B-A3B → ASI-Evolve  (Colab Pro / G4 Blackwell 98GB)

Run ASI-Evolve's Researcher/Engineer/Analyzer agents against a local Qwen3.6-35B-A3B served by `llama-server`, **or** against any OpenAI-compatible cloud endpoint (Anthropic OpenAI-compat, OpenAI, Together, etc.) by flipping `PROVIDER_MODE`.

No QuantClaw. No TurboQuant. Just the model + ASI-Evolve.

**Before running:** edit the `CONFIG` cell. For local mode, set `HF_TOKEN` as a Colab secret. For cloud mode, set the matching API-key secret (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`).

## 1. CONFIG (edit me)

In [ ]:
import os, pathlib, subprocess
from google.colab import userdata

# ---------- EDIT ----------
PROVIDER_MODE     = 'local'                 # 'local' | 'anthropic' | 'openai'

# Which ASI-Evolve experiment to run (must exist under experiments/<name>/)
EXPERIMENT_NAME   = 'circle_packing_demo'
EVOLVE_STEPS      = 10
SAMPLE_N          = 3

# Local (llama-server) settings
HF_MODEL_ID       = 'Qwen/Qwen3.6-35B-A3B'
QUANT_TYPE        = 'Q5_K_M'                # Q4_K_M, Q5_K_M, Q6_K, Q8_0, or f16
CTX_SIZE          = 131072
LLAMA_PORT        = 8081
LLAMACPP_SHA      = 'd006858316d4650bb4da0c6923294ccd741caefd'

# Cloud settings (only used if PROVIDER_MODE != 'local')
CLOUD_MODEL = {
    'anthropic': 'claude-opus-4-7',
    'openai':    'gpt-4.1',
}
CLOUD_BASE_URL = {
    'anthropic': 'https://api.anthropic.com/v1/',
    'openai':    'https://api.openai.com/v1/',
}
CLOUD_SECRET_NAME = {
    'anthropic': 'ANTHROPIC_API_KEY',
    'openai':    'OPENAI_API_KEY',
}

# Repo
ASI_EVOLVE_REPO = 'https://github.com/J-mazz/ASI-Evolve.git'
ASI_EVOLVE_REF  = 'main'
# --------------------------

assert PROVIDER_MODE in ('local', 'anthropic', 'openai'), f'bad PROVIDER_MODE: {PROVIDER_MODE!r}'

ROOT       = pathlib.Path('/content')
EVOLVE_DIR = ROOT / 'ASI-Evolve'
MODEL_DIR  = ROOT / 'models' / HF_MODEL_ID.split('/', 1)[1]
GGUF_F16   = MODEL_DIR / f'{MODEL_DIR.name}-f16.gguf'
GGUF_QUANT = MODEL_DIR / f'{MODEL_DIR.name}-{QUANT_TYPE}.gguf'
LLAMA_DIR  = ROOT / 'llama.cpp'
BUILD_DIR  = LLAMA_DIR / 'build-colab'

# HF token needed for local (download) mode.
if PROVIDER_MODE == 'local':
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Pull cloud key into env if selected.
if PROVIDER_MODE != 'local':
    key = userdata.get(CLOUD_SECRET_NAME[PROVIDER_MODE])
    assert key, f'Colab secret {CLOUD_SECRET_NAME[PROVIDER_MODE]} not set'
    os.environ['EVOLVE_API_KEY'] = key

cc = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                    capture_output=True, text=True).stdout.strip().split('\n')[0].replace('.', '')
CUDA_ARCH = cc or '80'
print(f'PROVIDER_MODE = {PROVIDER_MODE}')
print(f'CUDA arch     = sm_{CUDA_ARCH}')
print(f'Experiment    = {EXPERIMENT_NAME}  (steps={EVOLVE_STEPS})')


## 2. System deps

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y cmake ninja-build build-essential git-lfs pkg-config \
    libcurl4-openssl-dev
git lfs install --skip-smudge
pip install -q 'huggingface_hub[cli]>=0.25' sentencepiece protobuf numpy requests tqdm pyyaml

## 3. Clone ASI-Evolve + install its requirements

In [ ]:
!rm -rf {EVOLVE_DIR}
!git clone --depth 1 --branch {ASI_EVOLVE_REF} {ASI_EVOLVE_REPO} {EVOLVE_DIR}
assert (EVOLVE_DIR / 'main.py').exists(), 'ASI-Evolve clone failed'

req = EVOLVE_DIR / 'requirements.txt'
if req.exists():
    !pip install -q -r {req}
# Experiment-specific deps, if present
exp_req = EVOLVE_DIR / 'experiments' / EXPERIMENT_NAME / 'requirements.txt'
if exp_req.exists():
    !pip install -q -r {exp_req}

exp_cfg = EVOLVE_DIR / 'experiments' / EXPERIMENT_NAME / 'config.yaml'
assert exp_cfg.exists(), f'missing experiment config: {exp_cfg}'
print('ASI-Evolve ready')

## 4. (local only) Download Qwen3.6 + build llama.cpp + quantize

Cloud mode skips this entire section.

In [ ]:
if PROVIDER_MODE == 'local':
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    !huggingface-cli download {HF_MODEL_ID} \
        --local-dir {MODEL_DIR} \
        --local-dir-use-symlinks False \
        --exclude '*.bin' '*.pt' 'original/*'
    !du -sh {MODEL_DIR}
else:
    print(f'skip (cloud mode: {PROVIDER_MODE})')

In [ ]:
if PROVIDER_MODE == 'local':
    !rm -rf {LLAMA_DIR}
    !git clone https://github.com/ggerganov/llama.cpp.git {LLAMA_DIR}
    !git -C {LLAMA_DIR} checkout --quiet {LLAMACPP_SHA}

    # Install convert deps.
    conv_req = LLAMA_DIR / 'requirements' / 'requirements-convert_hf_to_gguf.txt'
    !pip install -q -r {conv_req}
else:
    print(f'skip (cloud mode: {PROVIDER_MODE})')

In [ ]:
if PROVIDER_MODE == 'local':
    import os
    os.environ['LLAMA_DIR'] = str(LLAMA_DIR)
    os.environ['BUILD_DIR'] = str(BUILD_DIR)
    os.environ['CUDA_ARCH'] = CUDA_ARCH
    !cmake -S $LLAMA_DIR -B $BUILD_DIR -G Ninja \
        -DCMAKE_BUILD_TYPE=Release \
        -DGGML_CUDA=ON \
        -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCH \
        -DLLAMA_BUILD_SERVER=ON
    !cmake --build $BUILD_DIR -j$(nproc) --target llama-quantize llama-server llama-cli
else:
    print(f'skip (cloud mode: {PROVIDER_MODE})')

In [ ]:
if PROVIDER_MODE == 'local':
    CONVERT = LLAMA_DIR / 'convert_hf_to_gguf.py'
    if not GGUF_F16.exists():
        !python {CONVERT} {MODEL_DIR} --outfile {GGUF_F16} --outtype f16
    if not GGUF_QUANT.exists() and QUANT_TYPE.lower() != 'f16':
        !{BUILD_DIR}/bin/llama-quantize {GGUF_F16} {GGUF_QUANT} {QUANT_TYPE}
    !ls -lh {GGUF_F16} {GGUF_QUANT if QUANT_TYPE.lower() != 'f16' else ''}
else:
    print(f'skip (cloud mode: {PROVIDER_MODE})')

## 5. (local only) Start llama-server

In [ ]:
import subprocess, time, socket, requests

llama_proc = None
LLAMA_LOG = ROOT / 'llama-server.log'

if PROVIDER_MODE == 'local':
    gguf = GGUF_QUANT if QUANT_TYPE.lower() != 'f16' else GGUF_F16
    cmd = [
        str(BUILD_DIR / 'bin' / 'llama-server'),
        '-m', str(gguf),
        '--host', '127.0.0.1',
        '--port', str(LLAMA_PORT),
        '--parallel', '1',
        '--ctx-size', str(CTX_SIZE),
        '-ngl', '99',              # offload all layers to GPU
    ]
    llama_proc = subprocess.Popen(cmd, stdout=LLAMA_LOG.open('wb'), stderr=subprocess.STDOUT)
    print(f'llama-server pid={llama_proc.pid}  log={LLAMA_LOG}')

    for _ in range(240):
        try:
            r = requests.get(f'http://127.0.0.1:{LLAMA_PORT}/health', timeout=2)
            if r.status_code == 200:
                print('llama-server HEALTHY'); break
        except Exception:
            pass
        time.sleep(1)
    else:
        raise RuntimeError(f'llama-server did not become healthy  see {LLAMA_LOG}')
else:
    print(f'skip (cloud mode: {PROVIDER_MODE})')

## 6. Write an ASI-Evolve config for this run

Clones the chosen experiment's `config.yaml`, overrides the `api` block to target the selected provider, saves as `config.notebook.yaml`.

In [ ]:
import yaml, copy

exp_dir = EVOLVE_DIR / 'experiments' / EXPERIMENT_NAME
src_cfg = yaml.safe_load((exp_dir / 'config.yaml').read_text())
cfg = copy.deepcopy(src_cfg)

if PROVIDER_MODE == 'local':
    cfg['api'] = {
        **cfg.get('api', {}),
        'provider': 'openai',                                  # OpenAI-compatible client
        'base_url': f'http://127.0.0.1:{LLAMA_PORT}/v1',
        'api_key':  'sk-no-key-required',
        'model':    HF_MODEL_ID.split('/', 1)[1],              # llama-server ignores, but keep descriptive
    }
else:
    cfg['api'] = {
        **cfg.get('api', {}),
        'provider': 'openai',
        'base_url': CLOUD_BASE_URL[PROVIDER_MODE],
        'api_key':  os.environ['EVOLVE_API_KEY'],
        'model':    CLOUD_MODEL[PROVIDER_MODE],
    }

# Force wandb offline for Colab (no network wandb login flow).
cfg.setdefault('logging', {}).setdefault('wandb', {})
cfg['logging']['wandb']['enabled'] = True
cfg['logging']['wandb']['offline'] = True

notebook_cfg = exp_dir / 'config.notebook.yaml'
notebook_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f'wrote {notebook_cfg}')
print('---\napi block:')
print(yaml.safe_dump({'api': cfg['api']}, sort_keys=False))

## 7. Smoke-test the provider endpoint

One-shot chat completion so we catch auth / URL / model-name issues *before* starting the evolve loop.

In [ ]:
import json, requests

api = cfg['api']
r = requests.post(
    api['base_url'].rstrip('/') + '/chat/completions',
    headers={'Authorization': f'Bearer {api["api_key"]}',
             'Content-Type':  'application/json'},
    json={
        'model': api['model'],
        'messages': [{'role': 'user', 'content': 'Reply with the single word: READY.'}],
        'max_tokens': 8,
        'temperature': 0,
    },
    timeout=120,
)
r.raise_for_status()
out = r.json()
print(json.dumps(out.get('usage', {}), indent=2))
print('reply:', out['choices'][0]['message']['content'])

## 8. Run ASI-Evolve

In [ ]:
import os
os.chdir(EVOLVE_DIR)
!python main.py \
    --config {notebook_cfg} \
    --experiment {EXPERIMENT_NAME} \
    --steps {EVOLVE_STEPS} \
    --sample-n {SAMPLE_N}

## 9. Teardown

In [ ]:
if llama_proc is not None:
    llama_proc.terminate()
    try:
        llama_proc.wait(timeout=10)
    except Exception:
        llama_proc.kill()
    print('llama-server stopped')